In [2]:
import numpy as np
import matplotlib.pyplot as plt
import random
import math
from scipy.optimize import minimize

In [3]:
#parameters
alpha_min = 0
alpha_max = 20
t_start = 0
t_end = 55
step = 0.01
num_neurons = 5

In [4]:
def normalize(x, x_min, x_max):
    return (x - x_min) / (x_max - x_min)

In [5]:
def I_input_advanced(t, ti, W):
    tau = 0.5
    return np.exp(-abs(t - ti)/tau)*W


def I_final(t,ti_list, W):
    suma = 0
    for u in ti_list:
        suma += I_input_advanced(t, u, W)
    return suma

In [6]:
def calculate_Hz(x):
    return alpha_min + x * (alpha_max - alpha_min)

In [7]:
def coding(x, t_start, t_end):
    freq = calculate_Hz(x)
    number_peaks = math.ceil((t_end - t_start) * freq)
    return number_peaks


In [8]:
def decoding(V_t):
    count = 0
    for i in range(len(V_t)-2):
        if V_t[i] < V_t[i + 1] and V_t[i + 1] > V_t[i + 2] and V_t[i+1] >= 30:
            count += 1
    return count

In [9]:
#we get the opposite function of calculating frequency -> so from frequency to get value from [0,1]
def scale(count):
    return (count - alpha_min) / (alpha_max - alpha_min)

In [10]:

a = 0.02
b = 0.2
c = -65 
d = 2


def dv_dt(v, u, I):
    return 0.04*v**2 + 5*v + 140 - u + I
    
def du_dt(v, u, a, b):
    return a*(v*b-u)

def derivatives(vec, I, a, b):
    return np.array([dv_dt(vec[0], vec[1], I), du_dt(vec[0], vec[1], a, b)])

def izikevich_model(ti_list, W, t_start, t_end, step):
    n = math.ceil((t_end-t_start)/step)
    t = np.linspace(t_start, t_end, n)
    listY = np.zeros([n, 2])
    vo = -65
    listY[0] = [-65, -13]
    for i in range(1, n):
        I = I_final(t[i], ti_list, W)
        listY[i] = listY[i-1] + step*derivatives(listY[i-1], I, a, b)
        if listY[i][0] >= 30:
            listY[i][0] = c
            listY[i][1] += d
    return listY.T[0]

def layer(num_neurons, ti_list, weights):


    output = izikevich_model(ti_list, weights[0], t_start, t_end, step)
    for i in range(1,num_neurons):
        output += izikevich_model(ti_list, weights[i], t_start, t_end, step)
    
    number_spikes = decoding(output)
    scaling = scale(number_spikes)
    return scaling

    

In [11]:
x_inputs = np.array([3,4,8,10,25,24])
x_final = np.array([normalize(x, min(x_inputs), max(x_inputs)) for x in x_inputs])
x_freq = np.array([calculate_Hz(x) for x in x_final])
print(x_freq)

[ 0.          0.90909091  4.54545455  6.36363636 20.         19.09090909]


In [13]:
true_output = [v*v for v in x_inputs]
final_true_output = [normalize(y, min(true_output), 
                                    max(true_output)) for y in true_output]

In [14]:
print(final_true_output)

[0.0, 0.011363636363636364, 0.08928571428571429, 0.14772727272727273, 1.0, 0.9204545454545454]


In [15]:
def general_layer(num_neurons, weights):
    scaling = []
    for fr in x_freq:
        ti_list = np.linspace(t_start, t_end, (t_end - t_start) * math.ceil(fr))
        scaling.append(layer(num_neurons, ti_list, weights))
    print(scaling)
    return scaling

In [18]:
weights = np.ones(num_neurons)
general_layer(5, weights)

[0.0, 0.0, 0.1, 0.15, 0.6, 0.6]


[0.0, 0.0, 0.1, 0.15, 0.6, 0.6]

In [16]:

def MSE(weights, y_true):
    suma = 0
    y_predicted = general_layer(num_neurons, weights)
    print(y_predicted)
    for i in range(len(y_true)):
        suma += (y_true[i] - y_predicted[i])**2
    return suma


In [ ]:
res = minimize(MSE, weights, method='nelder-mead',  args=(final_true_output), options={'xatol': 1e-8, 'disp': True})

[0.0, 0.0, 0.1, 0.15, 0.6, 0.6]
[0.0, 0.0, 0.1, 0.15, 0.6, 0.6]
